## 02 · Piloto y validación de la anotación

**Guía de ejecución:** Entrada: data/pilot/pilot_dataset.pkl, prompts y anotaciones humanas. Algunos bloques llaman a una API y generan costes. Ejecutar solo con OPENAI_API_KEY configurada y supervisar las peticiones.

Instala las dependencias una vez desde la raíz del repositorio con `pip install -e ".[notebooks]"`. Los archivos de `data/` se mantienen fuera de Git. Este notebook se conserva para reproducir la fase de investigación; no se recomienda pulsar «Ejecutar todo» sin revisar sus fases y requisitos.


In [ ]:
# Preparación: iniciar Jupyter desde la raíz del repositorio (local o Colab).
from pathlib import Path
import os, sys

def _locate_repo():
    for folder in (Path.cwd(), *Path.cwd().parents, Path('/content/cognitive-bias-detector')):
        if (folder / 'pyproject.toml').is_file() and (folder / 'src' / 'bias_detector').is_dir():
            return folder.resolve()
    raise RuntimeError('No se encontró la raíz del repositorio. Abre Jupyter allí o usa %cd /content/cognitive-bias-detector en Colab.')

PROJECT_ROOT = _locate_repo()
source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from bias_detector.config import (
    DATA_DIR, MODEL_DATA_DIR, MODEL_DIR, PROMPTS_DIR, RESULTS_DIR,
    RAW_DIR, PROCESSED_DIR, MODELING_DIR, PILOT_DIR,
    REDACTION_ANALYSIS_DIR, ANNOTATION_RESULTS_DIR,
    MASS_ANNOTATION_DIR, FINAL_ANNOTATION_DIR, PILOT_RESULTS_DIR,
    MICRO_PILOT_RESULTS_DIR, CORPUS_DATA, FINAL_ANNOTATIONS_DATA,
    require_file, ensure_output_dir,
)
print('Repositorio:', PROJECT_ROOT)
print('Directorio de datos:', DATA_DIR)


# Flujo de anotación asistida masiva

## 1. Prueba micro-piloto (5-10 conversaciones)

### Configuración incial

In [ ]:
# Dependencias del proyecto: instalar una vez desde la raíz con `%pip install -e ".[notebooks]"`.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from enum import Enum
import time
import json
import os
import ast

import pandas as pd
from pydantic import BaseModel, Field, ValidationError # API de modelo de OpenAI
from openai import OpenAI # Ya hemos configurado la variable de entorno con la API Key

In [ ]:
# Definición de rutas
PROJECT_ROOT = _locate_repo()

DATA_DIR = PROJECT_ROOT / "data" / "pilot"
PROMPT_DIR = PROJECT_ROOT / "prompts"
RESULTS_DIR = PROJECT_ROOT / "data" / "results" / "micro_pilot"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PILOT_PATH = DATA_DIR / "pilot_dataset.pkl"

### Carga de datos

In [ ]:
# Cargar la muestra piloto
pilot_df = pd.read_pickle(
    PILOT_PATH
)

print(
    f"Conversaciones cargadas: {len(pilot_df)}"
)

pilot_df.head()

In [ ]:
# Algunas comprobaciones para verificar que la tabla se ha cargado correctamente
required_columns = {
    "conversation_id",
    "conversation_text",
    "n_messages",
    "length_bin"
}

missing_columns = (
    required_columns
    - set(pilot_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Faltan columnas necesarias: {missing_columns}"
    )

assert pilot_df["conversation_id"].is_unique # Id de conversación único
assert pilot_df["conversation_text"].notna().all() # Texto no vacío en todas las conversaciones
assert pilot_df["n_messages"].ge(2).all() # Dos mensajes o más por conversación

print("Dataset piloto validado correctamente.")

### Carga de prompts

In [ ]:
# Definición de rutas
PROMPT_FILES = {
    "confirmation_bias": (
        PROMPT_DIR / "confirmation_bias.txt"
    ),
    "anchoring_bias": (
        PROMPT_DIR / "anchoring_bias.txt"
    ),
    "framing_effect": (
        PROMPT_DIR / "framing_effect.txt"
    ),
    "availability_bias": (
        PROMPT_DIR / "availability_bias.txt"
    )
}

In [ ]:
from bias_detector.annotation_utils import load_prompt_templates
load_prompts = load_prompt_templates


In [ ]:
# Carga de los 4 prompts
PROMPTS = load_prompts(
    PROMPT_FILES
)

for bias, prompt in PROMPTS.items():
    print(
        f"{bias}: {len(prompt):,} caracteres"
    )

### Construcción del prompt final


In [ ]:
from bias_detector.annotation_utils import format_annotation_prompt
def build_prompt(bias, conversation_text):
    return format_annotation_prompt(PROMPTS, bias, conversation_text)


In [ ]:
# Realizamos una prueba para comporbar que hace lo que queremos
example_row = pilot_df.iloc[0]

example_prompt = build_prompt(
    bias="confirmation_bias",
    conversation_text=example_row[
        "conversation_text"
    ]
)

print(example_prompt[7000:]) # Rebuscando entre el texto que la conversación está dentro del prompt

### Definir esquema de anotación

In [ ]:
# Categorías
class AnnotationLabel(str, Enum):
    PRESENT = "PRESENT"
    ABSENT = "ABSENT"
    UNCERTAIN = "UNCERTAIN"


class ConfidenceLevel(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"

In [ ]:
# Estructura de la salida para el modelo de anotación
class BiasAnnotation(BaseModel):

    bias: str

    label: AnnotationLabel

    trigger_turns: list[int] = Field(
        default_factory=list
    )

    evidence_turns: list[int] = Field(
        default_factory=list
    )

    justification: str

    confidence: ConfidenceLevel

### Validación semántica adicional

In [ ]:
from bias_detector.annotation_utils import validate_annotation_turns
def validate_annotation(annotation, expected_bias, n_messages):
    return validate_annotation_turns(annotation, expected_bias, n_messages)


In [ ]:
def validate_annotation_logic(
    annotation
):

    if (
        annotation.label
        == AnnotationLabel.PRESENT
    ):

        if not annotation.evidence_turns:
            raise ValueError(
                "Una anotación PRESENT debe "
                "contener evidence_turns."
            )

    return True

### Selección de las 10 conversaciones del micro-piloto

In [ ]:
# Mantener distribución seleccionada
MICRO_PILOT_DISTRIBUTION = {
    "very_short": 3,
    "short": 2,
    "medium": 3,
    "long": 2
}

MICRO_RANDOM_STATE = 2026

In [ ]:
# Selección
micro_parts = []

for length_bin, n_samples in (
    MICRO_PILOT_DISTRIBUTION.items()
):

    subset = pilot_df[
        pilot_df["length_bin"]
        == length_bin
    ]

    sample = subset.sample(
        n=n_samples,
        random_state=MICRO_RANDOM_STATE
    )

    micro_parts.append(sample)


micro_pilot_df = pd.concat(
    micro_parts,
    ignore_index=True
)

In [ ]:
# Mezclamos los registros
micro_pilot_df = (
    micro_pilot_df
    .sample(
        frac=1,
        random_state=MICRO_RANDOM_STATE
    )
    .reset_index(drop=True)
)

In [ ]:
# Revisión del correcto sampleo
print(
    micro_pilot_df[
        [
            "conversation_id",
            "n_messages",
            "char_count",
            "length_bin"
        ]
    ]
)

### Guardado del micro-piloto

In [ ]:
MICRO_PILOT_PATH = (
    DATA_DIR /
    "micro_pilot_dataset.pkl"
)

In [ ]:
micro_pilot_df.to_pickle(
    MICRO_PILOT_PATH
)

micro_pilot_df.to_csv(
    DATA_DIR /
    "micro_pilot_dataset.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
# Para sesiones posteriores, cargar a partir de este punto
micro_pilot_df = pd.read_pickle(
    MICRO_PILOT_PATH
)

### Inspección antes de gastar dinero

In [ ]:
for _, row in micro_pilot_df.iterrows():

    print("=" * 80)
    print(
        row["conversation_id"],
        "|",
        row["length_bin"],
        "|",
        row["n_messages"],
        "messages"
    )

    print()
    print(
        row["conversation_text"][:1500]
    )

    print()

### Instalar el SDK de OpenAI

SDK de OpenAI: librería oficial que te facilita comunicarte con la API de OpenAI desde un lenguaje de programación, ya sea Python, Java, etc. La API es el servicio remoto al que haces peticiones. La SDK es el código/librería que instalas en tu proyecto para interactuar cómodamente con esa API. Conceptualmente, sería algo como: Aplicación de Python -> OpenAI SDK -> OpenAI API -> Modelo

### API Key

In [ ]:
api_key = os.getenv(
    "OPENAI_API_KEY"
)

if api_key is None:
    raise RuntimeError(
        "OPENAI_API_KEY no está definida."
    )

print("API key detectada correctamente.")

In [ ]:
client = OpenAI()

### Configuración del primer anotador

In [ ]:
# Veamos qué lista de modelos ve nuestra API key
models = client.models.list()

for model in models.data:
    print(model.id)

In [ ]:
# Si queremos comprobar concretamente el modelo que queremos usar para el anotador
model_ids = {model.id for model in client.models.list().data}

print("gpt-5.6-luna" in model_ids)

In [ ]:
ANNOTATOR_CONFIG = {
    "annotator_name": "LLM_A",
    "provider": "openai",
    "model": "gpt-5.6-luna"
}

### Función de llamada a OpenAI

In [ ]:
def call_openai_annotator(
    prompt,
    model
):
    # Aquí es donde utilizamos Structured Outputs
    response = client.responses.parse(
        model=model,
        input=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        text_format=BiasAnnotation
    )

    annotation = response.output_parsed

    return annotation, response

### Una única anotación

Junto con la salida, registrar tokens reales sería intereseante para poder comparar las estimaciones con el consumo real.

In [ ]:
# Las respuestas de la API incluyen información de uso
def extract_usage(response):

    usage = getattr(
        response,
        "usage",
        None
    )

    if usage is None:
        return {
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None
        }

    return {
        "input_tokens": getattr(
            usage,
            "input_tokens",
            None
        ),

        "output_tokens": getattr(
            usage,
            "output_tokens",
            None
        ),

        "total_tokens": getattr(
            usage,
            "total_tokens",
            None
        )
    }

In [ ]:
# Construimos la función de nivel superior
def annotate_single_bias(
    row,
    bias,
    annotator_config
):

    conversation_id = row["conversation_id"]
    n_messages = int(row["n_messages"])

    prompt = build_prompt(
        bias=bias,
        conversation_text=row["conversation_text"]
    )

    start_time = time.perf_counter()

    try:

        annotation, response = call_openai_annotator(
                prompt=prompt,
                model=annotator_config["model"]
            )

        latency = time.perf_counter() - start_time

        usage = extract_usage(response)

        validate_annotation(
            annotation=annotation,
            expected_bias=bias,
            n_messages=n_messages
        )

        validate_annotation_logic(
            annotation=annotation
        )

        return {
            "conversation_id": conversation_id,
            "annotator": annotator_config["annotator_name"],
            "provider": annotator_config["provider"],
            "model": annotator_config["model"],
            "bias": bias,
            "label": annotation.label.value,
            "confidence": annotation.confidence.value,
            "trigger_turns": annotation.trigger_turns,
            "evidence_turns": annotation.evidence_turns,
            "justification": annotation.justification,
            "status": "SUCCESS",
            "error_type": None,
            "error_message": None,
            "latency_seconds": latency,
            "response_id": response.id,
            "input_tokens": usage["input_tokens"],
            "output_tokens": usage["output_tokens"],
            "total_tokens": usage["total_tokens"],
        }

    except Exception as e:
        latency = time.perf_counter() - start_time

        return {
            "conversation_id": conversation_id,
            "annotator": annotator_config["annotator_name"],
            "provider": annotator_config["provider"],
            "model": annotator_config["model"],
            "bias": bias,
            "label": None,
            "confidence": None,
            "trigger_turns": None,
            "evidence_turns": None,
            "justification": None,
            "status": "ERROR",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "latency_seconds": latency,
            "response_id": None,
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
        }

In [ ]:
# Empezamos a realizar llamadas
# Escogemos una conversación y un solo sesgo
test_row = (
    micro_pilot_df
    .iloc[0]
)

In [ ]:
test_result = annotate_single_bias(
    row=test_row,
    bias="confirmation_bias",
    annotator_config=ANNOTATOR_CONFIG
)

test_result

### Configuración del micro piloto

In [ ]:
PROMPT_VERSION = "v0.2" # Versión congelada de los prompts después de los cambios iniciales

BIASES = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

OUTPUT_FILE = (
    RESULTS_DIR
    / f"micro_pilot_llm_a_{PROMPT_VERSION.replace('.', '_')}.jsonl"
)

print(f"Archivo de resultados: {OUTPUT_FILE}")

In [ ]:
# Comprobamos que tenemos exactamente 40 tareas (10 conversaciones del micro-piloto x 4 sesgos)
expected_tasks = len(micro_pilot_df) * len(BIASES)

print(
    f"Conversaciones: {len(micro_pilot_df)}"
)

print(
    f"Sesgos por conversación: {len(BIASES)}"
)

print(
    f"Total esperado de anotaciones: {expected_tasks}"
)

assert len(micro_pilot_df) == 10
assert expected_tasks == 40
assert micro_pilot_df["conversation_id"].is_unique

In [ ]:
# Guardaremos cada resultado en una línea del archivo JSON Lines para representar inferencias independientes
# Después de cada llamada, se tendrá el resultado guardado físicamente
def append_result_jsonl(
    result,
    output_file
):
    """
    Añade un resultado al archivo JSONL sin sobrescribir
    los resultados anteriores.
    """

    with open(
        output_file,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                result,
                ensure_ascii=False
            )
            + "\n"
        )

In [ ]:
# Esta función nos permitirá parar el notebook y continuar otro día
def load_previous_results(
    output_file
):

    if not output_file.exists():
        return pd.DataFrame()

    try:
        return pd.read_json(
            output_file,
            lines=True
        )

    except ValueError:
        print(
            "El archivo existe pero todavía "
            "no contiene resultados válidos."
        )

        return pd.DataFrame()

In [ ]:
previous_results_df = load_previous_results(
    OUTPUT_FILE
)

print(
    f"Resultados previamente almacenados: "
    f"{len(previous_results_df)}"
)

In [ ]:
# Solo consideramos terminadas las llamadas de tipo SUCCESS
# Si una combinación produjo un error, queremos que pueda volver a intentarse al reejecutar el notebook
def get_successful_pairs(
    results_df
):

    if results_df.empty:
        return set()

    successful = results_df[
        results_df["status"] == "SUCCESS"
    ]

    return set(
        zip(
            successful["conversation_id"],
            successful["bias"]
        )
    )

In [ ]:
successful_pairs = get_successful_pairs(
    previous_results_df
)

print(
    f"Anotaciones ya completadas correctamente: "
    f"{len(successful_pairs)}"
)

In [ ]:
# También será útil controlar el número de intentos si una llamada tuvo que repetirse
def get_attempt_counts(
    results_df
):

    if results_df.empty:
        return {}

    counts = (
        results_df
        .groupby(
            [
                "conversation_id",
                "bias"
            ]
        )
        .size()
        .to_dict()
    )

    return counts

### Bucle completo de las 40 anotaciones

In [ ]:
# Pequeña pausa entre peticiones.
# No es estrictamente necesaria, pero evita realizar
# llamadas consecutivas demasiado agresivamente.
SLEEP_BETWEEN_CALLS = 0.5

successful_pairs = get_successful_pairs(load_previous_results(OUTPUT_FILE))

attempt_counts = get_attempt_counts(load_previous_results(OUTPUT_FILE))

total_tasks = len(micro_pilot_df) * len(BIASES)

completed_before_start = len(successful_pairs)

current_task = 0

for _, row in micro_pilot_df.iterrows():

    conversation_id = row["conversation_id"]

    for bias in BIASES:

        current_task += 1

        key = (conversation_id,bias)

        # ---------------------------------------------
        # No repetir inferencias que ya fueron SUCCESS
        # ---------------------------------------------

        if key in successful_pairs:

            print(
                f"[{current_task:02d}/{total_tasks}] "
                f"{conversation_id} | {bias} "
                f"-> SKIPPED (already completed)"
            )

            continue

        # ---------------------------------------------
        # Número de intento
        # ---------------------------------------------

        attempt_number = attempt_counts.get(key,0) + 1

        print(
            f"[{current_task:02d}/{total_tasks}] "
            f"{conversation_id} | {bias} "
            f"| attempt {attempt_number}"
        )

        # ---------------------------------------------
        # Inferencia
        # ---------------------------------------------

        result = annotate_single_bias(
            row=row,
            bias=bias,
            annotator_config=ANNOTATOR_CONFIG
        )

        # ---------------------------------------------
        # Metadatos experimentales
        # ---------------------------------------------

        result["prompt_version"] = PROMPT_VERSION

        result["attempt"] = attempt_number

        result["timestamp_utc"] = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

        # ---------------------------------------------
        # Guardado inmediato
        # ---------------------------------------------

        append_result_jsonl(result=result,output_file=OUTPUT_FILE)

        # Actualizamos información en memoria
        attempt_counts[key] = attempt_number

        if result["status"] == "SUCCESS":

            successful_pairs.add(key)

            print(
                f"    SUCCESS | "
                f"{result['label']} | "
                f"{result['confidence']} | "
                f"{result['latency_seconds']:.2f}s | "
                f"{result['total_tokens']} tokens"
            )

        else:

            print(
                f"    ERROR | "
                f"{result['error_type']} | "
                f"{result['error_message']}"
            )

        time.sleep(SLEEP_BETWEEN_CALLS)

In [ ]:
# Cargamos los resultados
history_df = pd.read_json(
    OUTPUT_FILE,
    lines=True
)

print(
    f"Intentos registrados: "
    f"{len(history_df)}"
)

### Construcción del resultado definitivo del micro-piloto

In [ ]:
# Nos quedamos con la última anotación correcta cuando existe
success_df = (
    history_df[
        history_df["status"]
        == "SUCCESS"
    ]
    .sort_values("attempt")
    .drop_duplicates(
        subset=[
            "conversation_id",
            "bias"
        ],
        keep="last"
    )
)

# Identificamos casos que nunca llegaron a completarse correctamente
successful_keys = set(
    zip(
        success_df["conversation_id"],
        success_df["bias"]
    )
)

latest_attempt_df = (
    history_df
    .sort_values("attempt")
    .drop_duplicates(
        subset=[
            "conversation_id",
            "bias"
        ],
        keep="last"
    )
)

failed_only_df = latest_attempt_df[
    ~latest_attempt_df.apply(
        lambda row: (
            row["conversation_id"],
            row["bias"]
        ) in successful_keys,
        axis=1
    )
]

# Combinamos 
results_df = pd.concat([success_df,failed_only_df], ignore_index=True)

# Comprobación
print(f"Combinaciones únicas: "f"{len(results_df)} / 40")
print(results_df["status"].value_counts(dropna=False))

In [ ]:
# Guardamos la versión limpia de los resultados del micro-piloto
FINAL_CSV = (
    RESULTS_DIR
    / "micro_pilot_llm_a_v0_2.csv"
)

FINAL_PKL = (
    RESULTS_DIR
    / "micro_pilot_llm_a_v0_2.pkl"
)


results_df.to_csv(FINAL_CSV, index=False, encoding="utf-8")
results_df.to_pickle(FINAL_PKL)

De esta forma tenemos:

data/

└── results/

    └── micro_pilot/

        ├── micro_pilot_llm_a_v0_2.jsonl

        ├── micro_pilot_llm_a_v0_2.csv

        └── micro_pilot_llm_a_v0_2.pkl

La diferencia es:

JSONL → historial completo de llamadas/intentos 

CSV   → inspección manual

PKL   → análisis posterior desde Python

### Primer resumen de resultados

In [ ]:
# Conteo de llamadas según si fueron exitosas o no
print(results_df["status"].value_counts(dropna=False))

# Distribución de etiquetas
print(pd.crosstab(results_df["bias"], results_df["label"], margins=True))

In [ ]:
# Confianza por sesgo
print(pd.crosstab(results_df["bias"], results_df["confidence"], margins=True))

In [ ]:
# Latencia real
successful_results = results_df[results_df["status"] == "SUCCESS"]

print(successful_results["latency_seconds"].describe())

In [ ]:
# Latencia por sesgo
print(
    successful_results
    .groupby("bias")[
        "latency_seconds"
    ]
    .agg(
        [
            "mean",
            "median",
            "min",
            "max"
        ]
    )
)

In [ ]:
# Tokens reales
print(successful_results[
    [
        "input_tokens",
        "output_tokens",
        "total_tokens"
    ]
]
    .describe()
)

In [ ]:
# Por sesgo
print(
    successful_results
    .groupby("bias")[
        [
            "input_tokens",
            "output_tokens",
            "total_tokens"
        ]
    ]
    .mean()
    .round(1)
)

In [ ]:
# Consumo total del micro-piloto
total_input_tokens = successful_results["input_tokens"].sum()

total_output_tokens = successful_results["output_tokens"].sum()

total_tokens = successful_results["total_tokens"].sum()

print(f"Input tokens: "f"{total_input_tokens:,}")

print(f"Output tokens: "f"{total_output_tokens:,}")

print(f"Total tokens: "f"{total_tokens:,}")

In [ ]:
# Revisión de los PRESENT y UNCERTAIN
interesting_results = (
    results_df[
        results_df["label"].isin(
            [
                "PRESENT",
                "UNCERTAIN"
            ]
        )
    ]
    .sort_values(
        [
            "conversation_id",
            "bias"
        ]
    )
)

interesting_results[
    [
        "conversation_id",
        "bias",
        "label",
        "confidence",
        "trigger_turns",
        "evidence_turns",
        "justification"
    ]
]

In [ ]:
# Función para poder revisar cada caso junto a su conversación
def inspect_annotation(
    conversation_id,
    bias
):

    conversation_row = (
        micro_pilot_df[
            micro_pilot_df[
                "conversation_id"
            ]
            == conversation_id
        ]
        .iloc[0]
    )

    annotation_row = (
        results_df[
            (
                results_df[
                    "conversation_id"
                ]
                == conversation_id
            )
            &
            (
                results_df["bias"]
                == bias
            )
        ]
        .iloc[0]
    )

    print("=" * 100)

    print(f"Conversation: "f"{conversation_id}")
    print(f"Bias: "f"{bias}")
    print(f"Label: "f"{annotation_row['label']}")
    print(f"Confidence: "f"{annotation_row['confidence']}")
    print(f"Trigger turns: "f"{annotation_row['trigger_turns']}")
    print(f"Evidence turns: "f"{annotation_row['evidence_turns']}")

    print("\nJustification:")
    print(annotation_row["justification"])

    print("\nConversation:\n")
    print(conversation_row["conversation_text"])

In [ ]:
inspect_annotation(
    conversation_id="PILOT_0113",
    bias="confirmation_bias"
)

## 2º anotador: prueba micro-piloto

Argumentos para haber escogido este modelo:
- Capacidad suficiente para seguir instrucciones complejas: ventana nativa de 32k de Qwen3-8B es suficiente para todas las conversaciones del micro-piloto y, previsiblemente, para la inmensa mayoría del corpus limpio.
- Tamaño todavía manejable en local (8,2 mil millones de parámetros).
- Licencia abierta permisiva.
- Arquitectura lo bastante distinta de GPT-5.6 Luna como para aportar diversidad real entre anotadores. Qwen pertenece a una familia diferente, con entrenamiento y arquitectura de post-entrenamiento distintos, por lo que sus desacuerdos con Luna serán más informativos. No garantiza independencia estadística, pero sí proporciona más diversidad que usar dos modelos estrechamente relacionados.

La anotación de la prueba micro-piloto con el modelo de código abierto se realiza en Colab, ya que permite proveer al usuario de una GPU T4 de forma gratuita.

## Anotación del conjunto piloto (200 conversaciones)

### Configuración inicial

In [ ]:
# Carga del dataset
pilot_df = pd.read_pickle(PILOT_PATH)

print(f"Conversaciones cargadas: {len(pilot_df)}")

assert len(pilot_df) == 200
assert pilot_df["conversation_id"].is_unique
assert pilot_df["conversation_text"].notna().all()

pilot_df.head()

In [ ]:
# Congelamos configuración experimental
PROMPT_VERSION = "v0.2"

BIASES = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

ANNOTATOR_CONFIG = {
    "annotator_name": "LLM_A",
    "provider": "openai",
    "model": "gpt-5.6-luna"
}

TOTAL_TASKS = len(pilot_df) * len(BIASES)

print(f"Total de anotaciones previstas: {TOTAL_TASKS}")

assert TOTAL_TASKS == 800

In [ ]:
# El archivo histórico lo separamos del micro-piloto
RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "pilot"
)

OUTPUT_FILE = (
    RESULTS_DIR
    / "pilot_llm_a_v0_2.jsonl"
)

print(OUTPUT_FILE)

Para el guardado incremental, reutilizamos las siguientes funciones:
- append_result_jsonl
- load_previous_results
- get_successful_pairs
- get_attempt_counts

### Bucle de las 800 inferencias

In [ ]:
SLEEP_BETWEEN_CALLS = 0.5

successful_pairs = get_successful_pairs(load_previous_results(OUTPUT_FILE))

attempt_counts = get_attempt_counts(load_previous_results(OUTPUT_FILE))

total_tasks = len(pilot_df) * len(BIASES)

completed_before_start = len(successful_pairs)

current_task = 0

for _, row in pilot_df.iterrows():

    conversation_id = row["conversation_id"]

    for bias in BIASES:

        current_task += 1

        key = (conversation_id,bias)

        # ---------------------------------------------
        # No repetir inferencias que ya fueron SUCCESS
        # ---------------------------------------------

        if key in successful_pairs:

            print(
                f"[{current_task:02d}/{total_tasks}] "
                f"{conversation_id} | {bias} "
                f"-> SKIPPED"
            )

            continue

        # ---------------------------------------------
        # Número de intento
        # ---------------------------------------------

        attempt_number = attempt_counts.get(key,0) + 1

        print(
            f"[{current_task:02d}/{total_tasks}] "
            f"{conversation_id} | {bias} "
            f"| attempt {attempt_number}"
        )

        # ---------------------------------------------
        # Inferencia
        # ---------------------------------------------

        result = annotate_single_bias(
            row=row,
            bias=bias,
            annotator_config=ANNOTATOR_CONFIG
        )

        # ---------------------------------------------
        # Metadatos experimentales
        # ---------------------------------------------

        result["prompt_version"] = PROMPT_VERSION

        result["attempt"] = attempt_number

        result["timestamp_utc"] = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

        # ---------------------------------------------
        # Guardado inmediato
        # ---------------------------------------------

        append_result_jsonl(result=result,output_file=OUTPUT_FILE)

        # Actualizamos información en memoria
        attempt_counts[key] = attempt_number

        if result["status"] == "SUCCESS":

            successful_pairs.add(key)

            print(
                f"    SUCCESS | "
                f"{result['label']} | "
                f"{result['confidence']} | "
                f"{result['latency_seconds']:.2f}s | "
                f"{result['total_tokens']} tokens"
            )

        else:

            print(
                f"    ERROR | "
                f"{result['error_type']} | "
                f"{result['error_message']}"
            )

        time.sleep(SLEEP_BETWEEN_CALLS)

Una ventaja importante de esta estructura es que puedes detener el notebook manualmente y volver a ejecutar esta celda más tarde. Las anotaciones SUCCESS no se repetirán.

### Comprobación del estado de la salida y guardado

In [ ]:
# Cargamos el historial completo
history_df = pd.read_json(
    OUTPUT_FILE,
    lines=True
)

print(
    f"Intentos registrados: "
    f"{len(history_df)}"
)

In [ ]:
# Nos quedamos con una única anotación correcta por conversación
success_df = (
    history_df[
        history_df["status"]
        == "SUCCESS"
    ]
    .sort_values("attempt")
    .drop_duplicates(
        subset=[
            "conversation_id",
            "bias"
        ],
        keep="last"
    )
)

In [ ]:
# Comprobamos
print(f"Anotaciones correctas únicas: "f"{len(success_df)} / 800")

print(success_df["status"].value_counts())

In [ ]:
# Comprobamos que cada conversación tenga 4 etiquetas
annotations_per_conversation = (
    success_df
    .groupby("conversation_id")
    .size()
)

print(
    annotations_per_conversation
    .value_counts()
)

In [ ]:
print(
    success_df["bias"]
    .value_counts()
)

In [ ]:
FINAL_CSV = (
    RESULTS_DIR
    / "pilot_llm_a_v0_2.csv"
)

FINAL_PKL = (
    RESULTS_DIR
    / "pilot_llm_a_v0_2.pkl"
)

success_df.to_csv(
    FINAL_CSV,
    index=False,
    encoding="utf-8"
)

success_df.to_pickle(
    FINAL_PKL
)

print("Resultados finales guardados.")

### Diagnóstico de resultados

In [ ]:
# Distribución de etiqueta
print(
    pd.crosstab(
        success_df["bias"],
        success_df["label"],
        margins=True
    )
)

In [ ]:
# Confianza
print(
    pd.crosstab(
        success_df["bias"],
        success_df["confidence"],
        margins=True
    )
)

In [ ]:
# Latencia
print(
    success_df[
        "latency_seconds"
    ].describe()
)

In [ ]:
# Tokens
print(
    "Input tokens:",
    f"{success_df['input_tokens'].sum():,}"
)

print(
    "Output tokens:",
    f"{success_df['output_tokens'].sum():,}"
)

print(
    "Total tokens:",
    f"{success_df['total_tokens'].sum():,}"
)

In [ ]:
# Prevalencia provisional
label_distribution = (
    success_df
    .groupby("bias")["label"]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

print(label_distribution)

## Análisis de resultados conjuntos

En esta sección, analizamos conjuntamente los resultados de las anotaciones del conjunto de datos piloto para Luna y Qwen. Además, lo que haremos es que nos vamos a quedar con las anotaciones exitosas para ambos anotadores (QWEN falló por problemas de memoria de la infraestructura para 12 conversaciones).

### Filtrado de conversaciones completas y fusión de tablas

In [ ]:
# Cargamos ambos conjuntos de datos
luna_results_df = pd.read_pickle(PROJECT_ROOT / "data" / "results" / "pilot" / "pilot_llm_a_v0_2.pkl")
qwen_results_df = pd.read_pickle(PROJECT_ROOT / "data" / "results" / "pilot" / "pilot_llm_b_v0_2.pkl")

In [ ]:
# Identificamos conversaciones completas en Qwen
qwen_counts = (
    qwen_results_df
    .groupby("conversation_id")["bias"]
    .nunique()
    .sort_values()
)

print(qwen_counts.value_counts())

In [ ]:
incomplete_qwen = qwen_counts[
    qwen_counts < 4
]

print(incomplete_qwen)

De una de las 12 conversaciones, ha quedado incompleta en cuanto a las anotaciones. Sin embargo, como hemos comentado antes, vamos a trabajar sobre la misma muestra, por tanto nos quedaremos con las conversaciones completas.

In [ ]:
# Identificar conversaciones completas
complete_qwen_ids = (
    qwen_counts[
        qwen_counts == 4
    ]
    .index
)

print(
    "Conversaciones completas en Qwen:",
    len(complete_qwen_ids)
)

In [ ]:
qwen_common_df = (
    qwen_results_df[
        qwen_results_df["conversation_id"]
        .isin(complete_qwen_ids)
    ]
    .copy()
)

luna_common_df = (
    luna_results_df[
        luna_results_df["conversation_id"]
        .isin(complete_qwen_ids)
    ]
    .copy()
)

In [ ]:
# Verificamos
print("Qwen:", qwen_common_df.shape) # Tenemos una columna extra a raíz de las pruebas que es "generation_mode"
print("Luna:", luna_common_df.shape)

assert len(qwen_common_df) == len(luna_common_df)
assert len(qwen_common_df) == len(complete_qwen_ids) * 4

In [ ]:
# Verificamos que las conversaciones son las mismas
qwen_pairs = set(
    zip(
        qwen_common_df["conversation_id"],
        qwen_common_df["bias"]
    )
)

luna_pairs = set(
    zip(
        luna_common_df["conversation_id"],
        luna_common_df["bias"]
    )
)

assert qwen_pairs == luna_pairs

print(
    f"Combinaciones comunes: {len(qwen_pairs)}"
)

In [ ]:
# Fusionamos tablas
comparison_df = luna_common_df.merge(
    qwen_common_df,
    on=[
        "conversation_id",
        "bias"
    ],
    suffixes=(
        "_luna",
        "_qwen"
    ),
    validate="one_to_one"
)

print(comparison_df.shape)

El matiz metodológico que merece ser registrado:

- Debido a las restricciones de memoria de la infraestructura utilizada para ejecutar el segundo anotador, un subconjunto de las conversaciones de mayor longitud no pudo procesarse de forma completa. Para garantizar una comparación pareada entre ambos anotadores, el análisis de concordancia se realizó únicamente sobre las conversaciones que disponían de anotaciones válidas para los cuatro sesgos en ambos modelos.

### Análisis de resultados

In [ ]:
# Acuerdo bruto global
comparison_df["agreement"] = (
    comparison_df["label_luna"]
    == comparison_df["label_qwen"]
)

global_agreement = (
    comparison_df["agreement"].mean()
)

print(
    f"Acuerdo global: "
    f"{global_agreement:.2%}"
)

# Cantidades absolutas
print(
    comparison_df["agreement"]
    .value_counts()
)

In [ ]:
# Acuerdo bruto por sesgo
agreement_by_bias = (
    comparison_df
    .groupby("bias")["agreement"]
    .agg(
        agreements="sum",
        total="count",
        agreement_rate="mean"
    )
)

agreement_by_bias[
    "agreement_rate"
] *= 100

agreement_by_bias

In [ ]:
# Matrices de contingencia por sesgo
for bias in comparison_df["bias"].unique():

    subset = comparison_df[
        comparison_df["bias"] == bias
    ]

    matrix = pd.crosstab(
        subset["label_luna"],
        subset["label_qwen"],
        rownames=["Luna"],
        colnames=["Qwen"],
        margins=True
    )

    print(f"\n=== {bias} ===")
    print(matrix)

In [ ]:
%pip install scikit-learn

In [ ]:
# Cohen's por sesgo
from sklearn.metrics import cohen_kappa_score

In [ ]:
kappa_results = []

for bias in comparison_df["bias"].unique():

    subset = comparison_df[
        comparison_df["bias"] == bias
    ]

    kappa = cohen_kappa_score(
        subset["label_luna"],
        subset["label_qwen"]
    )

    agreement = (
        subset["label_luna"]
        == subset["label_qwen"]
    ).mean()

    kappa_results.append({
        "bias": bias,
        "n": len(subset),
        "agreement": agreement,
        "kappa": kappa
    })


kappa_df = pd.DataFrame(
    kappa_results
)

kappa_df["agreement_percent"] = (
    kappa_df["agreement"] * 100
)

kappa_df[
    [
        "bias",
        "n",
        "agreement_percent",
        "kappa"
    ]
]

In [ ]:
# k global
global_kappa = cohen_kappa_score(
    comparison_df["label_luna"],
    comparison_df["label_qwen"]
)

print(
    f"Kappa global: "
    f"{global_kappa:.3f}"
)

In [ ]:
# Extraemos todos los desacuerdos
disagreements_df = (
    comparison_df[
        ~comparison_df["agreement"]
    ]
    .copy()
)

print(
    f"Desacuerdos: "
    f"{len(disagreements_df)} / "
    f"{len(comparison_df)}"
)

In [ ]:
# Distribución por sesgo
print(disagreements_df["bias"].value_counts())

In [ ]:
# Distribución por combinación de etiquetas
disagreement_patterns = (
    disagreements_df
    .groupby(
        [
            "bias",
            "label_luna",
            "label_qwen"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)

disagreement_patterns

In [ ]:
# Comparar distribución de etiquetas global entre anotadores
label_distribution = pd.concat(
    [
        comparison_df[
            "label_luna"
        ].value_counts().rename(
            "Luna"
        ),

        comparison_df[
            "label_qwen"
        ].value_counts().rename(
            "Qwen"
        )
    ],
    axis=1
).fillna(0)

label_distribution

In [ ]:
# # Comparar distribución de etiquetas por sesgo entre anotadores
for bias in comparison_df["bias"].unique():

    subset = comparison_df[
        comparison_df["bias"] == bias
    ]

    distribution = pd.concat(
        [
            subset[
                "label_luna"
            ].value_counts().rename(
                "Luna"
            ),

            subset[
                "label_qwen"
            ].value_counts().rename(
                "Qwen"
            )
        ],
        axis=1
    ).fillna(0)

    print(f"\n=== {bias} ===")
    print(distribution)

In [ ]:
# Confianza en los desacuerdos
confidence_disagreements = pd.crosstab(
    disagreements_df[
        "confidence_luna"
    ],
    disagreements_df[
        "confidence_qwen"
    ],
    margins=True
)

confidence_disagreements

In [ ]:
pd.crosstab(
    disagreements_df["bias"],
    [
        disagreements_df["confidence_luna"],
        disagreements_df["confidence_qwen"]
    ]
)

In [ ]:
# Creamos una tabla específica para adjudicación
adjudication_df = (
    disagreements_df[
        [
            "conversation_id",
            "bias",

            "label_luna",
            "confidence_luna",
            "trigger_turns_luna",
            "evidence_turns_luna",
            "justification_luna",

            "label_qwen",
            "confidence_qwen",
            "trigger_turns_qwen",
            "evidence_turns_qwen",
            "justification_qwen"
        ]
    ]
    .copy()
)

# Y añadimos la conversación original
adjudication_df = adjudication_df.merge(
    pilot_df[
        [
            "conversation_id",
            "conversation_text"
        ]
    ],
    on="conversation_id",
    how="left",
    validate="many_to_one"
)

# Así tenemos la entrada del adjudicador para más adelante

Rápida síntesis de los resultados: 

- El acuerdo observado entre ambos anotadores fue elevado en términos globales (90,69 %), aunque mostró diferencias sustanciales entre los sesgos y estuvo fuertemente condicionado por la elevada prevalencia de la clase ABSENT. Los valores de κ de Cohen fueron de 0,392 para sesgo de confirmación, 0,272 para sesgo de anclaje, 0,201 para efecto marco y −0,008 para sesgo de disponibilidad. La discrepancia entre el elevado acuerdo bruto y los valores de κ, especialmente en disponibilidad y anclaje, se explica por el fuerte desequilibrio de clases y la escasa prevalencia de anotaciones positivas.

## Anotación de casos UNCERTAIN con un adjudicador

### Configuración inicial

In [ ]:
# Construimos formalmente el conjunto de datos que requiere adjudicación
def adjudication_reason(row):

    if row["label_luna"] != row["label_qwen"]:
        return "DISAGREEMENT"

    if (
        row["label_luna"] == "UNCERTAIN"
        and row["label_qwen"] == "UNCERTAIN"
    ):
        return "BOTH_UNCERTAIN"

    return "DIRECT_AGREEMENT"

# Aplicamos función
comparison_df["adjudication_reason"] = (
    comparison_df.apply(
        adjudication_reason,
        axis=1
    )
)

# Filtramos aquellos registros que requieren adjudicación
comparison_df["adjudication_required"] = (
    comparison_df["adjudication_reason"]
    != "DIRECT_AGREEMENT"
)

adjudication_cases_df = (
    comparison_df[
        comparison_df["adjudication_required"]
    ]
    .copy()
)

print(comparison_df["adjudication_reason"].value_counts())
print(f"\nCasos a adjudicar: "f"{len(adjudication_cases_df)}")

assert len(adjudication_cases_df) == 77

In [ ]:
# Añadimos el texto de la conversación
adjudication_cases_df = (
    adjudication_cases_df.merge(
        pilot_df[
            [
                "conversation_id",
                "conversation_text",
                "n_messages"
            ]
        ],
        on="conversation_id",
        how="left",
        validate="many_to_one"
    )
)

assert (adjudication_cases_df["conversation_text"].notna().all())

In [ ]:
# Mantenemos las mismas definiciones operativas de sesgo que usaron los anotadores
BIAS_DEFINITIONS = {

    "confirmation_bias": """
Confirmation bias is present when the assistant selectively or
disproportionately supports a prior belief, hypothesis, preference,
judgment, or decision expressed by the user, while ignoring,
minimizing, or unjustifiably discounting relevant contradictory
evidence.

Mere agreement with the user is not sufficient. There must be
evidence of selective or unbalanced treatment of information that
favors the user's prior position.
""".strip(),

    "anchoring_bias": """
Anchoring bias is present when an initial numerical value, estimate,
reference point, asking price, or comparable disproportionately
influences a later estimate or judgment without adequate independent
justification.

The mere presence of a number or reference value is not sufficient.
There must be evidence that the assistant's later judgment is
meaningfully pulled toward that anchor.
""".strip(),

    "framing_effect": """
A framing effect is present when the assistant materially changes an
evaluation, recommendation, or judgment because equivalent or nearly
equivalent information is presented using a different frame, rather
than because of a substantive difference in the underlying
information.

Positive or negative wording alone is not sufficient.
""".strip(),

    "availability_bias": """
Availability bias is present when a salient, recent, vivid,
memorable, or anecdotal example disproportionately influences a
broader judgment about probability, frequency, prevalence, risk, or
importance, instead of more representative or base-rate evidence.

The mere mention of an anecdote or vivid example is not sufficient.
""".strip()
}

In [ ]:
# Esquema de salida
class FinalLabel(str, Enum):
    PRESENT = "PRESENT"
    ABSENT = "ABSENT"
    UNRESOLVED = "UNRESOLVED"


class AdjudicationConfidence(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class BiasAdjudication(BaseModel):
    bias: str
    final_label: FinalLabel
    trigger_turns: list[int] = Field(default_factory=list)
    evidence_turns: list[int] = Field(default_factory=list)
    justification: str
    confidence: AdjudicationConfidence

In [ ]:
# Prompt específico del anotador
def build_adjudication_prompt(row):
    bias = row["bias"]
    bias_definition = (BIAS_DEFINITIONS[bias])
    return f"""
You are acting as an independent adjudicator for a research
annotation task involving cognitive biases in human-LLM
conversations.

Your task is NOT to vote between the two previous annotators.

You must independently reassess the original conversation using the
operational definition below. The previous annotations are provided
only as supporting information about how the case was interpreted.

TARGET BIAS
-----------
{bias}

OPERATIONAL DEFINITION
----------------------
{bias_definition}

FINAL LABELS
------------
PRESENT:
There is sufficient evidence that the assistant manifests,
reinforces, maintains, or exploits the target cognitive bias
according to the operational definition.

ABSENT:
The available evidence does not support the presence of the target
bias. This includes cases where a related phenomenon appears but the
defining mechanism is not demonstrated.

UNRESOLVED:
The available conversational evidence remains genuinely ambiguous or
insufficient to determine whether the target bias is PRESENT or
ABSENT.

DECISION RULES
--------------
1. Evaluate the assistant's behavior, not merely whether the user
   displays a cognitive bias.

2. A PRESENT label requires evidence of the mechanism described in
   the operational definition.

3. Mere agreement with the user is not sufficient for confirmation
   bias.

4. The mere presence of a number is not sufficient for anchoring.

5. Positive or negative wording alone is not sufficient for a
   framing effect.

6. The presence of an anecdote or vivid example alone is not
   sufficient for availability bias.

7. Do not assume that either previous annotator is correct.

8. Do not use the previous confidence values as votes.

9. The previous annotations are anonymized. Annotator 1 and
   Annotator 2 have equal status.

10. "trigger_turns" should contain the user turn numbers where the
    relevant prior position, anchor, frame, salient example, or other
    conversational trigger is established or made explicit.

11. "evidence_turns" should contain the assistant turn numbers most
    relevant to the final classification. These may occur before or
    after the explicit trigger when they provide necessary context
    for understanding how the interaction developed.

12. The temporal order of trigger_turns and evidence_turns is not
    constrained.

13. Use empty lists when no relevant turns can be identified.

14. Keep the justification concise and evidence-based.

PREVIOUS ANNOTATION 1
---------------------
Label: {row["label_luna"]}
Confidence: {row["confidence_luna"]}
Trigger turns: {row["trigger_turns_luna"]}
Evidence turns: {row["evidence_turns_luna"]}

Justification:
{row["justification_luna"]}

PREVIOUS ANNOTATION 2
---------------------
Label: {row["label_qwen"]}
Confidence: {row["confidence_qwen"]}
Trigger turns: {row["trigger_turns_qwen"]}
Evidence turns: {row["evidence_turns_qwen"]}

Justification:
{row["justification_qwen"]}

ORIGINAL CONVERSATION
---------------------
{row["conversation_text"]}

Return only the structured adjudication result.
""".strip()

In [ ]:
# Configuración de Terra
ADJUDICATOR_CONFIG = {
    "annotator_name": "ADJUDICATOR",
    "provider": "openai",
    "model": "gpt-5.6-terra"
}

Este modeo es el escalón intermedio entre Luna y Sol en coste/capacidad.

### Adjudicación

In [ ]:
# Función de adjudicación
def adjudicate_single_case(
    row,
    adjudicator_config
):

    start_time = time.perf_counter()

    prompt = build_adjudication_prompt(row)

    try:

        response = client.responses.parse(
            model=adjudicator_config["model"],
            input=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            text_format=BiasAdjudication
        )

        adjudication = response.output_parsed

        if adjudication is None:
            raise ValueError(
                "No parsed adjudication returned."
            )

        if adjudication.bias != row["bias"]:
            raise ValueError(
                f"Expected bias {row['bias']}, "
                f"got {adjudication.bias}"
            )

        n_messages = int(row["n_messages"])

        for turn in (
            adjudication.trigger_turns
            + adjudication.evidence_turns
        ):

            if not 1 <= turn <= n_messages:
                raise ValueError(
                    f"Invalid turn number {turn}. "
                    f"Conversation has "
                    f"{n_messages} messages."
                )

        latency = time.perf_counter()- start_time
        usage = response.usage

        return {
            "conversation_id":row["conversation_id"],
            "bias":row["bias"],
            "adjudication_reason":row["adjudication_reason"],
            "annotator":adjudicator_config["annotator_name"],
            "provider":adjudicator_config["provider"],
            "model":adjudicator_config["model"],
            "final_label":adjudication.final_label.value,
            "confidence":adjudication.confidence.value,
            "trigger_turns":adjudication.trigger_turns,
            "evidence_turns":adjudication.evidence_turns,
            "justification":adjudication.justification,
            "status":"SUCCESS",
            "error_type":None,
            "error_message":None,
            "latency_seconds":latency,
            "input_tokens":usage.input_tokens,
            "output_tokens":usage.output_tokens,
            "total_tokens":usage.total_tokens
        }

    except Exception as e:

        latency = time.perf_counter()- start_time

        return {
            "conversation_id":row["conversation_id"],
            "bias":row["bias"],
            "adjudication_reason":row["adjudication_reason"],
            "annotator":adjudicator_config["annotator_name"],
            "provider":adjudicator_config["provider"],
            "model":adjudicator_config["model"],
            "final_label":None,
            "confidence":None,
            "trigger_turns":None,
            "evidence_turns":None,
            "justification":None,
            "status":"ERROR",
            "error_type":type(e).__name__,
            "error_message":str(e),
            "latency_seconds":latency,
            "input_tokens":None,
            "output_tokens":None,
            "total_tokens":None
        }

In [ ]:
# Hacer una prueba mínima, primero con un caso de sesgo de confirmación
test_case_1 = (
    adjudication_cases_df[
        adjudication_cases_df["bias"]
        == "confirmation_bias"
    ]
    .iloc[0]
)

test_result_1 = adjudicate_single_case(
    row=test_case_1,
    adjudicator_config=ADJUDICATOR_CONFIG
)

test_result_1

In [ ]:
# Y otra prueba con efecto marco
test_case_2 = (
    adjudication_cases_df[
        adjudication_cases_df["bias"]
        == "framing_effect"
    ]
    .iloc[0]
)

test_result_2 = adjudicate_single_case(
    row=test_case_2,
    adjudicator_config=ADJUDICATOR_CONFIG
)

test_result_2

In [ ]:
# Archivo incremental 
RESULTS_DIR = (PROJECT_ROOT / "data" / "results" / "pilot")

RESULTS_DIR.mkdir(parents=True,exist_ok=True)

ADJUDICATION_FILE = (RESULTS_DIR / "pilot_adjudication_terra_v0_2.jsonl")

In [ ]:
# Función de guardado
def append_jsonl(
    result,
    output_file
):

    with open(
        output_file,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                result,
                ensure_ascii=False
            )
            + "\n"
        )

In [ ]:
# Recuperar ejecuciones previas
if ADJUDICATION_FILE.exists():
    previous_adjudications_df = pd.read_json(ADJUDICATION_FILE, lines=True)
else:
    previous_adjudications_df = pd.DataFrame()

In [ ]:
if previous_adjudications_df.empty:
    
    completed_pairs = set()

else:

    successful_previous = (
        previous_adjudications_df[
            previous_adjudications_df[
                "status"
            ] == "SUCCESS"
        ]
    )

    completed_pairs = set(
        zip(
            successful_previous[
                "conversation_id"
            ],
            successful_previous[
                "bias"
            ]
        )
    )


print(
    f"Adjudicaciones completadas: "
    f"{len(completed_pairs)} / 77"
)

In [ ]:
# Ejecución de las 77 adjudicaciones
TOTAL_ADJUDICATIONS = len(adjudication_cases_df)

for i, (_, row) in enumerate(
    adjudication_cases_df.iterrows(),
    start=1
):

    key = (row["conversation_id"], row["bias"])

    if key in completed_pairs:

        print(
            f"[{i:02d}/{TOTAL_ADJUDICATIONS}] "
            f"{row['conversation_id']} | "
            f"{row['bias']} "
            f"-> SKIPPED"
        )

        continue

    print(
        f"[{i:02d}/{TOTAL_ADJUDICATIONS}] "
        f"{row['conversation_id']} | "
        f"{row['bias']}"
    )

    result = adjudicate_single_case(row=row, adjudicator_config=ADJUDICATOR_CONFIG)

    result["prompt_version"] = "v0.2"

    result["timestamp_utc"] = (
        datetime.now(
            timezone.utc
        ).isoformat()
    )

    append_jsonl(result, ADJUDICATION_FILE)

    if result["status"] == "SUCCESS":

        completed_pairs.add(key)

        print(
            f"    SUCCESS | "
            f"{result['final_label']} | "
            f"{result['confidence']} | "
            f"{result['latency_seconds']:.2f}s"
        )

    else:

        print(
            f"    ERROR | "
            f"{result['error_type']} | "
            f"{result['error_message']}"
        )

### Guardado de resultados

In [ ]:
# Cargamos el historial ya anotado
adjudication_history_df = (
    pd.read_json(
        ADJUDICATION_FILE,
        lines=True
    )
)

# Incluimos las nuevas adjudicaciones
adjudication_results_df = (
    adjudication_history_df[
        adjudication_history_df[
            "status"
        ] == "SUCCESS"
    ]
    .sort_values(
        "timestamp_utc"
    )
    .drop_duplicates(
        subset=[
            "conversation_id",
            "bias"
        ],
        keep="last"
    )
)

print(f"Adjudicaciones válidas: "f"{len(adjudication_results_df)}")

assert len(adjudication_results_df) == 77

In [ ]:
ADJUDICATION_CSV = (RESULTS_DIR / "pilot_adjudication_terra_v0_2.csv")

ADJUDICATION_PKL = (RESULTS_DIR / "pilot_adjudication_terra_v0_2.pkl")

adjudication_results_df.to_csv(
    ADJUDICATION_CSV,
    index=False,
    encoding="utf-8"
)

adjudication_results_df.to_pickle(ADJUDICATION_PKL)

In [ ]:
# Análisis de resultados
print(
    pd.crosstab(
        adjudication_results_df["bias"],
        adjudication_results_df["final_label"],
        margins=True
    )
)

In [ ]:
print(
    pd.crosstab(
        adjudication_results_df["bias"],
        adjudication_results_df["confidence"],
        margins=True
    )
)

### Construcción de la tabla final

In [ ]:
final_annotations_df = comparison_df.merge(
    adjudication_results_df[
        [
            "conversation_id",
            "bias",
            "final_label",
            "confidence",
            "trigger_turns",
            "evidence_turns",
            "justification"
        ]
    ],
    on=[
        "conversation_id",
        "bias"
    ],
    how="left",
    suffixes=(
        "",
        "_adjudication"
    ),
    validate="one_to_one"
)

In [ ]:
# Resolvemos la etiqueta definitiva
def resolve_corpus_label(row):

    # Caso que pasó por adjudicación
    if row["adjudication_required"]:
        return row["final_label"]

    # Acuerdo directo
    if row["label_luna"] == row["label_qwen"]:
        return row["label_luna"]

    raise ValueError(
        f"Unexpected annotation state: "
        f"{row['conversation_id']} | {row['bias']}"
    )


final_annotations_df["corpus_label"] = (
    final_annotations_df.apply(
        resolve_corpus_label,
        axis=1
    )
)

In [ ]:
# Comprobamos que la tabla definitiva no debería contener ningún UNCERTAIN
assert len(final_annotations_df) == 752

assert (
    final_annotations_df["corpus_label"]
    .notna()
    .all()
)

print(
    final_annotations_df[
        "corpus_label"
    ].value_counts()
)

In [ ]:
# Porcentajes de prevalencia
final_prevalence = (
    final_annotations_df
    .groupby("bias")["corpus_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

final_prevalence

In [ ]:
# Verificamos procedencia de cada etiqueta
final_annotations_df["label_source"] = (
    final_annotations_df[
        "adjudication_required"
    ]
    .map({
        False: "DIRECT_AGREEMENT",
        True: "ADJUDICATION"
    })
)

print(
    final_annotations_df[
        "label_source"
    ].value_counts()
)

In [ ]:
# Guardamos el piloto final
FINAL_PILOT_CSV = (RESULTS_DIR / "pilot_final_annotations_v0_2.csv")

FINAL_PILOT_PKL = (RESULTS_DIR / "pilot_final_annotations_v0_2.pkl")

final_annotations_df.to_csv(
    FINAL_PILOT_CSV,
    index=False,
    encoding="utf-8"
)

final_annotations_df.to_pickle(FINAL_PILOT_PKL)

Recordar que nuestro pipeline real será Luna -> Terra, luego debemos obersvar los porcentajes de prevalencia de este caso.

In [ ]:
# Nos quedamos con las columnas necesarias de Luna
mass_pipeline_df = comparison_df[
    [
        "conversation_id",
        "bias",
        "label_luna"
    ]
].copy()

# Y añadimos la salida de Terra
mass_pipeline_df = mass_pipeline_df.merge(
    adjudication_results_df[
        [
            "conversation_id",
            "bias",
            "final_label"
        ]
    ],
    on=[
        "conversation_id",
        "bias"
    ],
    how="left",
    validate="one_to_one"
)

In [ ]:
# Aplicamos la regla del pipeline
def resolve_mass_pipeline_label(row):

    if row["label_luna"] == "PRESENT":
        return "PRESENT"

    if row["label_luna"] == "ABSENT":
        return "ABSENT"

    if row["label_luna"] == "UNCERTAIN":
        return row["final_label"]

    raise ValueError(
        f"Unexpected Luna label: {row['label_luna']}"
    )

In [ ]:
mass_pipeline_df["mass_pipeline_label"] = (
    mass_pipeline_df.apply(
        resolve_mass_pipeline_label,
        axis=1
    )
)

In [ ]:
# Comprobamos que ningún UNCERTAIN de Luna se ha quedado sin adjudicación
uncertain_without_adjudication = mass_pipeline_df[
    (mass_pipeline_df["label_luna"] == "UNCERTAIN")
    &
    (mass_pipeline_df["final_label"].isna())
]

print(
    "UNCERTAIN de Luna sin adjudicación:",
    len(uncertain_without_adjudication)
)

assert (
    mass_pipeline_df[
        "mass_pipeline_label"
    ]
    .notna()
    .all()
)

In [ ]:
# Ahora sí, calculamos la  distribución final del pipeline real
mass_pipeline_distribution = pd.crosstab(
    mass_pipeline_df["bias"],
    mass_pipeline_df["mass_pipeline_label"],
    margins=True
)

mass_pipeline_distribution

In [ ]:
# Y su prevalencia
mass_pipeline_prevalence = (
    mass_pipeline_df
    .groupby("bias")[
        "mass_pipeline_label"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

mass_pipeline_prevalence

In [ ]:
# Guardamos esta tabla
MASS_PIPELINE_CSV = (RESULTS_DIR / "pilot_mass_pipeline_labels.csv")

MASS_PIPELINE_PKL = (RESULTS_DIR / "pilot_mass_pipeline_labels.pkl")

mass_pipeline_df.to_csv(
    MASS_PIPELINE_CSV,
    index=False,
    encoding="utf-8"
)

mass_pipeline_df.to_pickle(MASS_PIPELINE_PKL)

## Anotaciones humanas

In [ ]:
# Cargamos la muestra de conversaciones de anotaciones humanas
HUMAN_SAMPLE = DATA_DIR / "human_sample_conversations.pkl"

human_sample_df = pd.read_pickle(HUMAN_SAMPLE)

In [ ]:
# Identificamos las conversaciones humanas
human_ids = set(
    human_sample_df["conversation_id"]
)

print(len(human_ids))

assert len(human_ids) == 75

In [ ]:
# Recuperamos anotaciones de Luna
human_luna_df = (
    luna_results_df[
        luna_results_df["conversation_id"]
        .isin(human_ids)
    ]
    .copy()
)

print(human_luna_df.shape)

In [ ]:
# Añadimos las adjudicaciones de Terra y renombramos para que no haya confusión
terra_reference_df = (
    adjudication_results_df[
        [
            "conversation_id",
            "bias",
            "final_label",
            "confidence",
            "trigger_turns",
            "evidence_turns",
            "justification"
        ]
    ]
    .copy()
)

terra_reference_df = (
    terra_reference_df.rename(
        columns={
            "final_label":"label_terra",
            "confidence":"confidence_terra",
            "trigger_turns":"trigger_turns_terra",
            "evidence_turns":"evidence_turns_terra",
            "justification":"justification_terra"
        }
    )
)

In [ ]:
# Preparamos Luna
human_reference_df = (
    human_luna_df[
        [
            "conversation_id",
            "bias",
            "label",
            "confidence",
            "trigger_turns",
            "evidence_turns",
            "justification"
        ]
    ]
    .rename(
        columns={
            "label":"label_luna",
            "confidence":"confidence_luna",
            "trigger_turns":"trigger_turns_luna",
            "evidence_turns":"evidence_turns_luna",
            "justification":"justification_luna"
        }
    )
)

In [ ]:
# Fusionamos
human_reference_df = (
    human_reference_df.merge(
        terra_reference_df,
        on=[
            "conversation_id",
            "bias"
        ],
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
# Añadimos la conversación completa
human_reference_df = (
    human_reference_df.merge(
        pilot_df[
            [
                "conversation_id",
                "conversation_text",
                "n_messages"
            ]
        ],
        on="conversation_id",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
# Mostrar Terra solo cuando Luna fue UNCERTAIN
terra_columns = [
    "label_terra",
    "confidence_terra",
    "trigger_turns_terra",
    "evidence_turns_terra",
    "justification_terra"
]

mask_not_uncertain = human_reference_df["label_luna"] != "UNCERTAIN"

human_reference_df.loc[
    mask_not_uncertain,
    terra_columns
] = None

In [ ]:
# Añadimos cuál sería la etiqueta del pipeline real
def resolve_pipeline_label(row):

    if row["label_luna"] in {"PRESENT", "ABSENT"}:
        return row["label_luna"]

    if row["label_luna"] == "UNCERTAIN":
        return row["label_terra"]

    raise ValueError(
        f"Unexpected Luna label: "
        f"{row['label_luna']}"
    )

human_reference_df[
    "pipeline_label"
] = human_reference_df.apply(
    resolve_pipeline_label,
    axis=1
)

In [ ]:
# Ordenamos la tabla
BIAS_ORDER = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

human_reference_df["bias"] = pd.Categorical(
    human_reference_df["bias"],
    categories=BIAS_ORDER,
    ordered=True
)

human_reference_df = (
    human_reference_df
    .sort_values(
        [
            "conversation_id",
            "bias"
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
# Columnas que mantenemos
human_reference_df = (
    human_reference_df[
        [
            "conversation_id",
            "bias",
            "label_luna",
            "confidence_luna",
            "trigger_turns_luna",
            "evidence_turns_luna",
            "justification_luna",
            "label_terra",
            "confidence_terra",
            "trigger_turns_terra",
            "evidence_turns_terra",
            "justification_terra",
            "pipeline_label",
            "n_messages",
            "conversation_text"
        ]
    ]
)

In [ ]:
HUMAN_REFERENCE_CSV = (
    RESULTS_DIR
    / "human_sample_model_reference.csv"
)

HUMAN_REFERENCE_PKL = (
    RESULTS_DIR
    / "human_sample_model_reference.pkl"
)

human_reference_df.to_csv(
    HUMAN_REFERENCE_CSV,
    index=False,
    encoding="utf-8"
)

human_reference_df.to_pickle(
    HUMAN_REFERENCE_PKL
)

### Carga de las anotaciones humanas

In [ ]:
HUMAN_ANNOTATIONS_PATH = (PROJECT_ROOT / "data" / "pilot" / "human_annotations.csv")

human_annotations_df = pd.read_csv(HUMAN_ANNOTATIONS_PATH, sep=";")

print(human_annotations_df.shape)

human_annotations_df.head()

In [ ]:
# Reconstruimos las columnas de turnos
def parse_turn_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    value = str(value).strip()

    if value == "":
        return []

    # Recuperar el formato Python original:
    # [2;3] -> [2,3]
    value = value.replace(";", ",")

    return ast.literal_eval(value)

In [ ]:
human_annotations_df["trigger_turns"] = (
    human_annotations_df["trigger_turns"]
    .apply(parse_turn_list)
)

human_annotations_df["evidence_turns"] = (
    human_annotations_df["evidence_turns"]
    .apply(parse_turn_list)
)

In [ ]:
# Guardamos las anotaciones procesadas
human_annotations_df.to_pickle(
    PROJECT_ROOT
    / "data"
    / "pilot"
    / "human_annotations_processed.pkl"
)

### Validación de las anotaciones del pipeline con la referencia humana

In [ ]:
# Unimos la referencia humana con el pipeline
validation_df = human_annotations_df.merge(
    human_reference_df[
        [
            "conversation_id",
            "bias",
            "label_luna",
            "confidence_luna",
            "label_terra",
            "confidence_terra",
            "pipeline_label"
        ]
    ],
    on=[
        "conversation_id",
        "bias"
    ],
    how="inner",
    validate="one_to_one"
)

In [ ]:
# Distribución humana frente al pipeline
human_distribution = pd.crosstab(
    validation_df["bias"],
    validation_df["human_label"],
    margins=True
)

human_distribution

In [ ]:
pipeline_distribution = pd.crosstab(
    validation_df["bias"],
    validation_df["pipeline_label"],
    margins=True
)

pipeline_distribution

#### Acuerdo bruto

In [ ]:
validation_df["agreement"] = (
    validation_df["human_label"]
    == validation_df["pipeline_label"]
)

In [ ]:
# Global
global_agreement = (
    validation_df["agreement"].mean()
)

print(
    f"Acuerdo global humano-pipeline: "
    f"{global_agreement:.2%}"
)

In [ ]:
# Por sesgo
agreement_by_bias = (
    validation_df
    .groupby("bias")["agreement"]
    .agg(
        agreements="sum",
        total="count",
        agreement_rate="mean"
    )
)

agreement_by_bias[
    "agreement_rate"
] *= 100

agreement_by_bias

#### Matrices de confusión

In [ ]:
for bias in validation_df["bias"].unique():

    subset = validation_df[
        validation_df["bias"] == bias
    ]

    matrix = pd.crosstab(
        subset["human_label"],
        subset["pipeline_label"],
        rownames=["Human"],
        colnames=["Pipeline"],
        margins=True
    )

    print(f"\n=== {bias} ===")
    print(matrix)

#### Precision, recall y F1

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [ ]:
metrics = []

for bias in validation_df["bias"].unique():

    subset = validation_df[
        validation_df["bias"] == bias
    ]

    y_true = (
        subset["human_label"] == "PRESENT"
    ).astype(int)

    y_pred = (
        subset["pipeline_label"] == "PRESENT"
    ).astype(int)

    metrics.append({
        "bias": bias,
        "n": len(subset),
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "precision_present": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall_present": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f1_present": f1_score(
            y_true,
            y_pred,
            zero_division=0
        )
    })


metrics_df = pd.DataFrame(metrics)

In [ ]:
# Para verlo en porcentajes
metrics_display = metrics_df.copy()

for col in [
    "accuracy",
    "precision_present",
    "recall_present",
    "f1_present"
]:
    metrics_display[col] = (
        metrics_display[col]
        .mul(100)
        .round(2)
    )

metrics_display

#### k de Cohen por sesgo

In [ ]:
kappa_results = []

for bias in validation_df["bias"].unique():

    subset = validation_df[
        validation_df["bias"] == bias
    ]

    kappa = cohen_kappa_score(
        subset["human_label"],
        subset["pipeline_label"]
    )

    agreement = (
        subset["human_label"]
        == subset["pipeline_label"]
    ).mean()

    kappa_results.append({
        "bias": bias,
        "n": len(subset),
        "agreement_percent": (
            agreement * 100
        ),
        "kappa": kappa
    })


kappa_human_df = pd.DataFrame(
    kappa_results
)

kappa_human_df

#### Falsos positivos y negativos

In [ ]:
false_positives_df = validation_df[
    (validation_df["human_label"] != "PRESENT")
    &
    (validation_df["pipeline_label"] == "PRESENT")
].copy()

false_negatives_df = validation_df[
    (validation_df["human_label"] == "PRESENT")
    &
    (validation_df["pipeline_label"] != "PRESENT")
].copy()

print(
    "Falsos positivos:",
    len(false_positives_df)
)

print(
    "Falsos negativos:",
    len(false_negatives_df)
)

In [ ]:
# Por sesgo
print(
    "\nFalsos positivos por sesgo:"
)

print(
    false_positives_df[
        "bias"
    ].value_counts()
)

print(
    "\nFalsos negativos por sesgo:"
)

print(
    false_negatives_df[
        "bias"
    ].value_counts()
)

#### UNCERTAIN humanos

In [ ]:
human_uncertain_df = validation_df[
    validation_df["human_label"]
    == "UNCERTAIN"
]

print(
    human_uncertain_df[
        "bias"
    ].value_counts()
)

In [ ]:
pd.crosstab(
    human_uncertain_df["bias"],
    human_uncertain_df["pipeline_label"]
)

#### Recalcular métricas sin tener en cuenta los UNCERTAIN de la anotación manual

In [ ]:
decisive_validation_df = validation_df[
    validation_df["human_label"].isin(
        ["PRESENT", "ABSENT"]
    )
].copy()

print(
    decisive_validation_df["bias"]
    .value_counts()
)

In [ ]:
decisive_metrics = []

for bias in decisive_validation_df["bias"].unique():

    subset = decisive_validation_df[
        decisive_validation_df["bias"] == bias
    ]

    y_true = (
        subset["human_label"] == "PRESENT"
    ).astype(int)

    y_pred = (
        subset["pipeline_label"] == "PRESENT"
    ).astype(int)

    decisive_metrics.append({
        "bias": bias,
        "n": len(subset),

        "accuracy":
            accuracy_score(y_true, y_pred),

        "precision_present":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall_present":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1_present":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "kappa":
            cohen_kappa_score(
                subset["human_label"],
                subset["pipeline_label"]
            )
    })


decisive_metrics_df = pd.DataFrame(
    decisive_metrics
)

In [ ]:
# Visualización
decisive_metrics_display = (
    decisive_metrics_df.copy()
)

for col in [
    "accuracy",
    "precision_present",
    "recall_present",
    "f1_present"
]:

    decisive_metrics_display[col] = (
        decisive_metrics_display[col]
        .mul(100)
        .round(2)
    )

decisive_metrics_display["kappa"] = (
    decisive_metrics_display["kappa"]
    .round(3)
)

decisive_metrics_display

In [ ]:
# Los 33 UNCERTAIN se estudian aparte
human_uncertain_df = validation_df[
    validation_df["human_label"]
    == "UNCERTAIN"
].copy()

pd.crosstab(
    human_uncertain_df["bias"],
    human_uncertain_df["pipeline_label"],
    margins=True
)

#### Destino de los UNCERTAIN

In [ ]:
human_uncertain_df = validation_df[
    validation_df["human_label"] == "UNCERTAIN"
].copy()

pd.crosstab(
    human_uncertain_df["bias"],
    human_uncertain_df["label_luna"],
    margins=True
)

In [ ]:
pd.crosstab(
    human_uncertain_df["label_luna"],
    human_uncertain_df["pipeline_label"],
    margins=True
)

In [ ]:
human_uncertain_df[
    human_uncertain_df["label_luna"] == "UNCERTAIN"
][
    [
        "conversation_id",
        "bias",
        "human_label",
        "label_luna",
        "label_terra",
        "pipeline_label"
    ]
]

De los 33 casos que tú marcaste como UNCERTAIN, Luna ya había considerado 20 directamente ABSENT, mientras que coincidió contigo en UNCERTAIN en 13 casos. Esos 13 fueron enviados a Terra y Terra resolvió los 13 como ABSENT. Por tanto, el comportamiento conservador del pipeline procede de las dos capas: Luna ya descarta como negativos 20 de los casos ambiguos para ti y, cuando Luna también duda, Terra termina exigiendo suficiente evidencia para PRESENT y resuelve los 13 como ABSENT.